## Week 8
### Final Milestone
Nana Noda-Morgan  
DSC 670-T301: Advanced Uses of Generative AI  
Professor Neugebauer  
March 3, 2026


Building upon my previous milestone, I will be using the trained model (ft:gpt-4.1-2025-04-14:personal:recipe-ingredient=classification:D8YBJCJA).


At the end of my previous milestone, I identified two issues. First, the detect_changes() function I had created used simple string matching, which was not working when the adapted recipe included rephrased ingredients (e.g., "1 carrot (thinly cut)" vs "1 large carrot, cut into thin matchsticks"). And secondly, the compute_drift() function was also not working, and returning 0, even though there were clearly ingredients that were changed. After examining the code again, I realized that I had a mistake in my function with the for loop, and it was causing the counts to always be 0.

In this final milestone, I'm going to try to make the app more robust by replacing the detect_changes() function with a model based approach, using GPT to semantically match the ingredients and classify each one has being removed, swapped, or unchanged. I will keep the compute_drift() function with the fix.

Also in my first step, I'm goign to add a new function to remove any extra info from the ingredients, such as quantity, unites, and preparation notes.

In [38]:
##First Import and set up variables

from openai import OpenAI
from pydantic import BaseModel
import re
import json

client = OpenAI()

#Model to get recipes and its adaptation based on zipcode
GPT_MODEL      = "gpt-5"
#My fine-tuned model to classify ingredients
FT_MODEL       = "ft:gpt-4.1-2025-04-14:personal:recipe-ingredient-classification:D8YBJCJA"
#Model to use in place of detect_change()
JUDGE_MODEL    = "gpt-4.1-mini-2025-04-14" 

def normalize(s: str) -> str:
    """Lowercase and remove spaces for consistent matching"""
    return re.sub(r"\s+", " ", s.strip().lower())


def strip_quantity(ingredient: str) -> str:
    """
    Strips leading quantities, units, and prep notes from an ingredient string.
    
    Examples:
      "1 Gobo (burdock root, Sasagaki cut)"  -> "gobo (burdock root)"
      "2 1/2 Tbsp Soy Sauce"                 -> "soy sauce"
      "1/4 cup Dashi (or water)"             -> "dashi (or water)"
    """
    s = ingredient.strip()
    s = re.sub(r"^\d+\s+\d+/\d+\s+", "", s)   # "2 1/2 "
    s = re.sub(r"^\d+/\d+\s+",        "", s)   # "1/2 "
    s = re.sub(r"^\d+\s*",            "", s)   # "1 "
    units = r"(?:Tbsp|tbsp|tsp|cup|cups|oz|lb|lbs|g|kg|ml|cloves?|stalks?|pieces?|cans?)\s+"
    s = re.sub(r"^" + units, "", s, flags=re.IGNORECASE)
    prep_keywords = r"thinly|cut|sliced|chopped|diced|minced|grated|julienned|sasagaki|peeled|trimmed"
    s = re.sub(r"\s*\([^)]*(?:" + prep_keywords + r")[^)]*\)", "", s, flags=re.IGNORECASE)
    return normalize(s)

def clean_ingredient(s: str) -> str:
    #Remove links like ([site.com](https://...)) from ingredient lists provided by ChatGPT
    return re.sub(r"\s*\(\[.*?\]\(.*?\)\)", "", s).strip()

Now this next chunk contains unchanged code form the preivous milestone that extracts the recipe and gets the adaptation based on zip.

In [34]:
class RecipeExtract(BaseModel):
    title: str
    servings: str | None = None
    prep_time: str | None = None
    cook_time: str | None = None
    total_time: str | None = None
    ingredients: list[str]
    instructions: list[str]


def get_original_recipe(url: str) -> RecipeExtract:
    """Fetch and parse the original recipe from a URL"""
    prompt = f"""
    Open this recipe page and extract the recipe details:
    URL: {url}

    Return:
    - title
    - servings
    - prep_time, cook_time, total_time (if available)
    - ingredients (as a list, keep each ingredient on one line exactly as written)
    - instructions (numbered)
    Return in JSON.
    """
    resp = client.responses.parse(
        model=GPT_MODEL,
        tools=[{"type": "web_search"}],
        reasoning={"effort": "low"},
        input=prompt,
        text_format=RecipeExtract,
    )
    return resp.output_parsed


def get_adapted_recipe(url: str, zip_code: str) -> RecipeExtract:
    """
    Adapt the recipe for ingredients available in the given zip code.
    Instructs the model to keep ingredient names consistent where no substitution is needed.
    """
    prompt = f"""
    Open this recipe page and review the recipe details:
    URL: {url}

    I am located in zip code {zip_code}. Look up grocery stores in my zip code and check 
    which ingredients from the recipe are available to me.

    Important instructions:
    - For ingredients that ARE available, keep the ingredient name exactly as it appears in the recipe.
    - For ingredients that are NOT available, substitute with what is available and clearly note the substitution.
    - Return the full updated ingredient list and any instruction changes needed.
    Return in JSON.
    """
    resp = client.responses.parse(
        model=GPT_MODEL,
        tools=[{"type": "web_search"}],
        reasoning={"effort": "low"},
        input=prompt,
        text_format=RecipeExtract,
    )
    adapted = resp.output_parsed

    #Remove inline URLs that are added to ingredients
    adapted.ingredients = [clean_ingredient(ing) for ing in adapted.ingredients]

    return adapted

Now this is the ingredient role classification using the fine-tuned model. This is also unchanged from the previous milestone.

In [24]:
def get_ingredient_roles(recipe_title: str, ingredients: list[str]) -> dict[str, str]:
    """
    Call the fine-tuned model to classify each ingredient as core, supporting, or optional.
    Returns a dict mapping normalized ingredient name -> role.
    """
    stripped = [strip_quantity(ing) for ing in ingredients]
    ingredient_lines = "\n".join(f"- {ing}" for ing in stripped)

    system_msg = (
        "You label each ingredient in a recipe as core, supporting, or optional. "
        "Respond with one ingredient per line in the format 'ingredient name: role'."
    )
    user_msg = f"Recipe title: {recipe_title}\nIngredients:\n{ingredient_lines}"

    resp = client.responses.create(
        model=FT_MODEL,
        input=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
    )

    text = resp.output[0].content[0].text.strip()

    roles: dict[str, str] = {}
    for line in text.splitlines():
        if ":" not in line:
            continue
        name, role = line.split(":", 1)
        roles[normalize(name)] = role.strip().lower()
    return roles

Now, I will be using a gpt model to detect ingredients that have changed. Instead of using string matching, the model is prompted to return a structed JSON object mapping each original ingredit to one of: unchanged, swapped, or removed. I believe this approach makes the app more robust to handle the natrual variation that comes with recipes.

In [28]:
def detect_changes_with_model(
    original_ingredients: list[str],
    adapted_ingredients: list[str],
) -> dict[str, str]:
    """
    Uses a language model to compare the original and adapted ingredients.
    Returns a dict mapping original ingredient name as 'unchanged' | 'swapped' | 'removed'.
    """
    original_list = "\n".join(f"- {ing}" for ing in original_ingredients)
    adapted_list  = "\n".join(f"- {ing}" for ing in adapted_ingredients)

    prompt = f"""
You are a recipe ingredient comparison assistant.

Compare the ORIGINAL ingredient list to the ADAPTED ingredient list.
For each ingredient in the ORIGINAL list, determine its status in the ADAPTED list:
- "unchanged": the same ingredient is present (even if phrased slightly differently)
- "swapped": a different ingredient was substituted for it
- "removed": the ingredient was dropped with no substitute

ORIGINAL ingredients:
{original_list}

ADAPTED ingredients:
{adapted_list}

Respond ONLY with a valid JSON object. Keys are the original ingredient names (exactly as listed above).
Values are one of: "unchanged", "swapped", or "removed".
Do not include any explanation or extra text — only the JSON object.
"""

    resp = client.responses.create(
        model=JUDGE_MODEL,
        input=[{"role": "user", "content": prompt}],
    )

    raw = resp.output[0].content[0].text.strip()

    #I read a Medium article about fenced code blocks and how sometimes responses 
    #include this even if instructed not to, so adding a few extra lines to ensure I'm only getting what I asked for
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    parsed: dict[str, str] = json.loads(raw)

    #Normalize so they match the role dict keys later on
    return {strip_quantity(k): v.strip().lower() for k, v in parsed.items()}

Now I am going to include the drift calculation, which is mostly the same as the previous milestone, but fixed the for loop so the counts are properly being added.

In [13]:
def compute_drift(
    roles: dict[str, str],
    change_status: dict[str, str],
    recipe_title: str | None = None,
) -> dict:
    """
    Computes the drift level of the adapted recipe compared to the original.
    Drift rules:
      LOW    — 0 core changed AND <= 2 supporting changed
      MEDIUM — 1 core changed AND <= 3 supporting changed
               OR 0 core changed AND 3-5 supporting changed
      HIGH   — 2+ core changed
               OR 1+ core removed entirely
               OR 1 core changed AND 4+ supporting changed
               OR 6+ supporting changed
    """
    core_changed      = 0
    core_removed      = 0
    supporting_changed = 0
    optional_changed  = 0

    #Add up all changes first before making any drift decision
    for name, role in roles.items():
        status = change_status.get(name, "unchanged")

        if role == "core":
            if status != "unchanged":
                core_changed += 1
                if status == "removed":
                    core_removed += 1
        elif role == "supporting":
            if status != "unchanged":
                supporting_changed += 1
        elif role == "optional":
            if status != "unchanged":
                optional_changed += 1

    #Drift classification
    if core_changed == 0 and supporting_changed <= 2:
        drift = "low"
    elif (
        (core_changed == 1 and supporting_changed <= 3)
        or (core_changed == 0 and 3 <= supporting_changed <= 5)
    ):
        drift = "medium"
    elif (
        core_changed >= 2
        or core_removed >= 1
        or (core_changed == 1 and supporting_changed >= 4)
        or supporting_changed >= 6
    ):
        drift = "high"
    else:
        drift = "medium"  #fallback

    title = recipe_title or "this recipe"

    if drift == "low":
        message = (
            f"Drift: LOW — {title} is still very close to the original recipe. "
            "All core ingredients are unchanged and only a few supporting ingredients were swapped."
        )
    elif drift == "medium":
        message = (
            f"Drift: MEDIUM — {title} is still recognizable as the same dish, "
            "but at least one core ingredient or several supporting ingredients have been changed. "
            "It is still worth making, but know that it will taste slightly different from the original."
        )
    else:
        message = (
            f"Drift: HIGH — {title} has changed significantly. "
            "Multiple core and/or supporting ingredients have been removed or substituted. "
            "It may still taste good, but it is no longer the original dish. "
            "You can choose a different recipe or think of this as an 'inspired by' version."
        )

    return {
        "drift_level":        drift,
        "core_changed":       core_changed,
        "core_removed":       core_removed,
        "supporting_changed": supporting_changed,
        "optional_changed":   optional_changed,
        "message":            message,
    }

Finally, the last function that puts it all together...

In [30]:
def analyze_recipe_for_zip(url: str, zip_code: str) -> dict:
    """
    Full process:
    1. Fetch original recipe
    2. Adapt recipe for what's available in the given zip code
    3. Classify ingredient roles using fine-tuned model
    4. Detect ingredient changes using model-based comparison
    5. Compute and report recipe drift
    """

    print("Fetching original recipe...")
    original = get_original_recipe(url)

    print(f"Adapting recipe for zip code {zip_code}...")
    adapted = get_adapted_recipe(url, zip_code)

    print("Classifying ingredient roles...")
    roles = get_ingredient_roles(
        recipe_title=original.title,
        ingredients=original.ingredients,
    )

    print("Detecting ingredient changes...")
    changes = detect_changes_with_model(
        original_ingredients=original.ingredients,
        adapted_ingredients=adapted.ingredients,
    )

    print("Computing drift...\n")
    drift_info = compute_drift(
        roles=roles,
        change_status=changes,
        recipe_title=original.title,
    )

    #Show results
    print("=" * 50)
    print(f"  {original.title}")
    print("=" * 50)

    print("\n--- Original Ingredients ---")
    for ing in original.ingredients:
        role = roles.get(strip_quantity(ing), "unknown")
        print(f"  [{role:10s}] {ing}")

    print(f"\n--- Adapted Ingredients (ZIP: {zip_code}) ---")
    for ing in adapted.ingredients:
        print(f"  {ing}")

    print("\n--- Ingredient Change Summary ---")
    for ing in original.ingredients:
        key = strip_quantity(ing)
        role   = roles.get(key, "unknown")
        status = changes.get(key, "unknown")
        marker = "  " if status == "unchanged" else "! "
        print(f"  {marker}[{role:10s}] {ing} → {status}")

    print("\n--- Should You Make This Recipe? ---")
    print(drift_info["message"])
    print(
        f"\n  core_changed={drift_info['core_changed']}, "
        f"core_removed={drift_info['core_removed']}, "
        f"supporting_changed={drift_info['supporting_changed']}, "
        f"optional_changed={drift_info['optional_changed']}"
    )

    return {
        "original": original,
        "adapted":  adapted,
        "roles":    roles,
        "changes":  changes,
        "drift":    drift_info,
    }

Now I'm going to test it out and see how it works!

In [40]:
url      = "https://www.japanesecooking101.com/kinpira-gobo-recipe/"
zip_code = "80758"

result = analyze_recipe_for_zip(url, zip_code)

Fetching original recipe...
Adapting recipe for zip code 80758...
Classifying ingredient roles...
Detecting ingredient changes...
Computing drift...

  Kinpira Gobo Recipe

--- Original Ingredients ---
  [core      ] 1 Gobo (burdock root, Sasagaki cut)
  [supporting] 1 carrot (thinly cut)
  [supporting] 1 Tbsp oil
  [supporting] 1/4 cup Dashi (or water)
  [core      ] 2 1/2 Tbsp Soy Sauce
  [supporting] 1 Tbsp sugar
  [supporting] 1 Tbsp Sake
  [supporting] 1 Tbsp Mirin
  [optional  ] 1 Tbsp sesame seeds
  [optional  ] Shichimi Togarashi (red hot pepper)

--- Adapted Ingredients (ZIP: 80758) ---
  1 carrot (thinly cut)
  1 Tbsp oil
  1/4 cup Dashi (or water) — SUB if Dashi not available: use 1/4 cup low-sodium chicken or vegetable broth (common grocery stock)
  2 1/2 Tbsp Soy Sauce
  1 Tbsp sugar
  1 Tbsp Sake — SUB if not available: use 1 Tbsp water or dry sherry if you have it; otherwise omit and add 1/2 Tbsp extra Soy Sauce
  1 Tbsp Mirin — SUB if not available: mix 1 Tbsp sugar wit

I'm really happy with how this turned out! I'm going to now move onto adapting this to be able to publish using streamlit. I'm assumign I'll have to make a few adaptations as I figure that step out. 

## References
  
OpenAI. (2023). *ChatGPT 5.1*  [Large language model]. https://chat.openai.com/chat  
  
OpenAI Developers. (n.d.). *List fine-tuning checkpoints.* Open AI. https://developers.openai.com/api/reference/resources/fine_tuning/subresources/jobs/subresources/checkpoints/methods/list

## Appendix 

Below is the ChatGPT 5.1 Prompt and response used to generate dataset. After I got the list, I added it to a script as you saw in my notebook above to add the system prompt and split the ingredients into user content and assistant content which had the roles, before creating the training.jsonl and validation.jsonl files. As you saw above in the notebook, I did also go through each recipe that was provided and clean it up or fix classifications to better align with my knowledge.

**Prompt:**
Help generate a list following this exact example and format of at least 30 recipe examples that can be used as a training dataset for fine-tuning, and a separate list that contains at least 10 recipe examples that can be used as a validation dataset. The recipes should all be Japanese recipes. You can use my example as the first recipe in the training dataset list.

And to this prompt, I attached a .jsonl file with one example:  
  
{"messages": [  
  {"role": "system", "content": "You are an expert Japanese cooking assistant. Given a recipe title and a list of ingredients, you label each ingredient as core, supporting, or optional."},  
  {"role": "user", "content": "Recipe title: Miso Soup (Tofu & Wakame)\nIngredients:\n- dashi stock\n- miso paste\n- silken tofu\n- dried wakame seaweed\n- scallions (green onion)\n- kombu (kelp) for dashi\n- katsuobushi (bonito flakes) for dashi"},  
  {"role": "assistant", "content": "dashi stock: core\nmiso paste: core\nsilken tofu: supporting\ndried wakame seaweed: supporting\nscallions (green onion): optional\nkombu (kelp) for dashi: supporting\nkatsuobushi (bonito flakes) for dashi: supporting"}  
]}    

 
**This was the full response:**
  
\# 30 training recipes  
training_recipes = [  
    {  
        "title": "Miso Soup (Tofu & Wakame)",
        "ingredients": [
            ("dashi stock", "core"),
            ("miso paste", "core"),
            ("silken tofu", "supporting"),
            ("dried wakame seaweed", "supporting"),
            ("scallions (green onion)", "optional"),
            ("kombu (kelp) for dashi", "supporting"),
            ("katsuobushi (bonito flakes) for dashi", "supporting"),
        ],
    },
    {
        "title": "Tonkatsu (Pork Cutlet)",
        "ingredients": [
            ("pork loin cutlets", "core"),
            ("panko breadcrumbs", "core"),
            ("all-purpose flour", "supporting"),
            ("egg", "supporting"),
            ("salt", "supporting"),
            ("black pepper", "supporting"),
            ("neutral frying oil", "supporting"),
            ("tonkatsu sauce", "supporting"),
            ("shredded cabbage", "optional"),
            ("lemon wedges", "optional"),
            ("karashi (Japanese mustard)", "optional"),
        ],
    },
    {
        "title": "Chicken Karaage",
        "ingredients": [
            ("boneless chicken thighs", "core"),
            ("soy sauce", "supporting"),
            ("sake", "supporting"),
            ("grated ginger", "supporting"),
            ("grated garlic", "supporting"),
            ("potato starch (or cornstarch)", "core"),
            ("all-purpose flour", "supporting"),
            ("neutral frying oil", "supporting"),
            ("lemon wedges", "optional"),
            ("Japanese mayonnaise", "optional"),
        ],
    },
    {
        "title": "Oyakodon (Chicken and Egg Rice Bowl)",
        "ingredients": [
            ("steamed short-grain rice", "core"),
            ("boneless chicken thighs", "core"),
            ("egg", "core"),
            ("onion", "supporting"),
            ("dashi stock", "supporting"),
            ("soy sauce", "supporting"),
            ("mirin", "supporting"),
            ("sugar", "supporting"),
            ("scallions or mitsuba", "optional"),
        ],
    },
    {
        "title": "Gyudon (Beef Rice Bowl)",
        "ingredients": [
            ("steamed short-grain rice", "core"),
            ("thinly sliced beef", "core"),
            ("onion", "supporting"),
            ("dashi stock", "supporting"),
            ("soy sauce", "core"),
            ("mirin", "supporting"),
            ("sugar", "supporting"),
            ("sake", "supporting"),
            ("beni shoga (pickled red ginger)", "optional"),
            ("soft-cooked egg", "optional"),
            ("scallions", "optional"),
        ],
    },
    {
        "title": "Okonomiyaki (Osaka-Style)",
        "ingredients": [
            ("all-purpose flour", "core"),
            ("dashi or water for batter", "supporting"),
            ("egg", "core"),
            ("finely shredded cabbage", "core"),
            ("thinly sliced pork belly", "supporting"),
            ("okonomiyaki sauce", "supporting"),
            ("Japanese mayonnaise", "optional"),
            ("aonori (seaweed flakes)", "optional"),
            ("katsuobushi (bonito flakes)", "optional"),
            ("pickled red ginger (beni shoga)", "optional"),
            ("neutral frying oil", "supporting"),
            ("tenkasu (tempura bits)", "optional"),
        ],
    },
    {
        "title": "Nikujaga (Meat and Potato Stew)",
        "ingredients": [
            ("thinly sliced beef or pork", "core"),
            ("potatoes", "core"),
            ("onion", "supporting"),
            ("carrot", "supporting"),
            ("dashi or water", "supporting"),
            ("soy sauce", "core"),
            ("mirin", "supporting"),
            ("sugar", "supporting"),
            ("sake", "supporting"),
            ("green peas or snow peas", "optional"),
        ],
    },
    {
        "title": "Yakisoba (Stir-Fried Noodles)",
        "ingredients": [
            ("yakisoba noodles", "core"),
            ("pork belly slices", "supporting"),
            ("cabbage", "supporting"),
            ("carrot", "supporting"),
            ("onion", "supporting"),
            ("bean sprouts", "optional"),
            ("yakisoba sauce", "core"),
            ("neutral frying oil", "supporting"),
            ("aonori (seaweed flakes)", "optional"),
            ("beni shoga (pickled red ginger)", "optional"),
            ("katsuobushi (bonito flakes)", "optional"),
        ],
    },
    {
        "title": "Onigiri (Salmon Rice Ball)",
        "ingredients": [
            ("steamed short-grain rice", "core"),
            ("grilled salted salmon", "core"),
            ("nori sheets", "supporting"),
            ("salt for seasoning rice", "supporting"),
            ("sesame seeds", "optional"),
            ("furikake seasoning", "optional"),
        ],
    },
    {
        "title": "Tonkotsu Ramen (Simplified)",
        "ingredients": [
            ("ramen noodles", "core"),
            ("rich pork-based broth (tonkotsu-style)", "core"),
            ("chashu (braised pork belly)", "supporting"),
            ("soft-boiled marinated egg", "supporting"),
            ("menma (seasoned bamboo shoots)", "optional"),
            ("scallions (green onions)", "optional"),
            ("nori sheets", "optional"),
            ("sesame seeds", "optional"),
            ("pork fat or aroma oil", "supporting"),
            ("tare seasoning", "core"),
        ],
    },
    {
        "title": "Teriyaki Chicken",
        "ingredients": [
            ("boneless chicken thighs", "core"),
            ("soy sauce", "core"),
            ("mirin", "core"),
            ("sake", "supporting"),
            ("sugar", "supporting"),
            ("grated ginger", "supporting"),
            ("neutral frying oil", "supporting"),
            ("steamed rice", "supporting"),
            ("scallions", "optional"),
            ("sesame seeds", "optional"),
        ],
    },
    {
        "title": "Salmon Teriyaki",
        "ingredients": [
            ("salmon fillets", "core"),
            ("soy sauce", "core"),
            ("mirin", "core"),
            ("sake", "supporting"),
            ("sugar", "supporting"),
            ("grated ginger", "supporting"),
            ("neutral frying oil", "supporting"),
            ("steamed rice", "supporting"),
            ("lemon wedges", "optional"),
        ],
    },
    {
        "title": "Tamagoyaki (Rolled Omelette)",
        "ingredients": [
            ("eggs", "core"),
            ("dashi or water", "supporting"),
            ("sugar", "supporting"),
            ("soy sauce", "supporting"),
            ("mirin", "supporting"),
            ("neutral frying oil", "supporting"),
            ("grated daikon radish", "optional"),
            ("soy sauce for serving", "optional"),
        ],
    },
    {
        "title": "Katsudon (Pork Cutlet Rice Bowl)",
        "ingredients": [
            ("steamed short-grain rice", "core"),
            ("pork loin cutlet (tonkatsu)", "core"),
            ("egg", "core"),
            ("onion", "supporting"),
            ("dashi stock", "supporting"),
            ("soy sauce", "supporting"),
            ("mirin", "supporting"),
            ("sugar", "supporting"),
            ("scallions", "optional"),
        ],
    },
    {
        "title": "Mixed Tempura",
        "ingredients": [
            ("shrimp", "core"),
            ("kabocha squash", "supporting"),
            ("sweet potato", "supporting"),
            ("eggplant", "supporting"),
            ("green beans", "supporting"),
            ("tempura batter (flour, egg, water)", "core"),
            ("neutral frying oil", "core"),
            ("tentsuyu dipping sauce", "supporting"),
            ("grated daikon radish", "optional"),
            ("grated ginger", "optional"),
        ],
    },
    {
        "title": "Zaru Soba (Cold Buckwheat Noodles)",
        "ingredients": [
            ("soba noodles", "core"),
            ("mentsuyu dipping sauce", "core"),
            ("water or dashi for dilution", "supporting"),
            ("nori strips", "optional"),
            ("scallions", "optional"),
            ("wasabi", "optional"),
            ("grated daikon radish", "optional"),
        ],
    },
    {
        "title": "Kitsune Udon",
        "ingredients": [
            ("udon noodles", "core"),
            ("dashi broth", "core"),
            ("soy sauce", "supporting"),
            ("mirin", "supporting"),
            ("aburaage (seasoned fried tofu)", "core"),
            ("scallions", "optional"),
            ("kamaboko fish cake", "supporting"),
            ("shichimi togarashi", "optional"),
        ],
    },
    {
        "title": "Takoyaki",
        "ingredients": [
            ("takoyaki batter (flour, dashi, egg)", "core"),
            ("octopus pieces", "core"),
            ("beni shoga (pickled red ginger)", "supporting"),
            ("tenkasu (tempura bits)", "supporting"),
            ("green onion", "supporting"),
            ("takoyaki sauce", "core"),
            ("Japanese mayonnaise", "optional"),
            ("aonori (seaweed flakes)", "optional"),
            ("katsuobushi (bonito flakes)", "optional"),
            ("neutral frying oil", "supporting"),
        ],
    },
    {
        "title": "Chirashi Sushi",
        "ingredients": [
            ("sushi rice", "core"),
            ("sashimi-grade salmon", "core"),
            ("sashimi-grade tuna", "core"),
            ("ikura (salmon roe)", "supporting"),
            ("tamagoyaki strips", "supporting"),
            ("cucumber", "supporting"),
            ("nori strips", "optional"),
            ("shiso leaves", "optional"),
            ("pickled ginger", "optional"),
            ("wasabi", "optional"),
        ],
    },
    {
        "title": "Japanese Beef Curry Rice",
        "ingredients": [
            ("thinly sliced beef", "core"),
            ("onions", "supporting"),
            ("carrots", "supporting"),
            ("potatoes", "supporting"),
            ("Japanese curry roux blocks", "core"),
            ("neutral frying oil", "supporting"),
            ("water or stock", "supporting"),
            ("steamed rice", "supporting"),
            ("fukujinzuke pickles", "optional"),
        ],
    },
    {
        "title": "Katsu Curry (Pork Cutlet Curry)",
        "ingredients": [
            ("pork loin cutlet (tonkatsu)", "core"),
            ("Japanese curry sauce", "core"),
            ("steamed rice", "core"),
            ("shredded cabbage", "optional"),
            ("fukujinzuke pickles", "optional"),
        ],
    },
    {
        "title": "Shabu-Shabu",
        "ingredients": [
            ("thinly sliced beef", "core"),
            ("napa cabbage", "supporting"),
            ("shungiku (chrysanthemum greens)", "supporting"),
            ("tofu", "supporting"),
            ("enoki mushrooms", "supporting"),
            ("shiitake mushrooms", "supporting"),
            ("negi (Japanese leek)", "supporting"),
            ("kombu for broth", "supporting"),
            ("water", "supporting"),
            ("ponzu sauce", "supporting"),
            ("goma dare (sesame dipping sauce)", "supporting"),
            ("steamed rice", "optional"),
        ],
    },
    {
        "title": "Sukiyaki",
        "ingredients": [
            ("thinly sliced beef", "core"),
            ("shirataki noodles", "supporting"),
            ("tofu", "supporting"),
            ("napa cabbage", "supporting"),
            ("negi (Japanese leek)", "supporting"),
            ("shiitake mushrooms", "supporting"),
            ("enoki mushrooms", "supporting"),
            ("warishita sauce", "core"),
            ("raw egg for dipping", "optional"),
        ],
    },
    {
        "title": "Miso-Glazed Eggplant (Nasu Dengaku)",
        "ingredients": [
            ("Japanese eggplant", "core"),
            ("miso paste", "core"),
            ("sugar", "supporting"),
            ("mirin", "supporting"),
            ("sake", "supporting"),
            ("neutral frying oil", "supporting"),
            ("sesame seeds", "optional"),
            ("scallions", "optional"),
        ],
    },
    {
        "title": "Spinach Goma-ae",
        "ingredients": [
            ("spinach", "core"),
            ("toasted sesame seeds", "core"),
            ("soy sauce", "supporting"),
            ("sugar", "supporting"),
            ("mirin", "supporting"),
            ("dashi", "supporting"),
            ("salt", "supporting"),
        ],
    },
    {
        "title": "Takikomi Gohan (Mixed Rice)",
        "ingredients": [
            ("Japanese short-grain rice", "core"),
            ("dashi", "core"),
            ("soy sauce", "supporting"),
            ("mirin", "supporting"),
            ("sake", "supporting"),
            ("chicken thighs", "supporting"),
            ("carrots", "supporting"),
            ("burdock root", "supporting"),
            ("shiitake mushrooms", "supporting"),
            ("aburaage (fried tofu pouch)", "supporting"),
            ("scallions", "optional"),
        ],
    },
    {
        "title": "Oyako Udon (Chicken and Egg Udon Soup)",
        "ingredients": [
            ("udon noodles", "core"),
            ("chicken thighs", "core"),
            ("egg", "core"),
            ("onion", "supporting"),
            ("dashi broth", "core"),
            ("soy sauce", "supporting"),
            ("mirin", "supporting"),
            ("scallions", "optional"),
            ("nori strips", "optional"),
        ],
    },
    {
        "title": "Karaage Bento Rice Bowl",
        "ingredients": [
            ("steamed rice", "core"),
            ("chicken karaage pieces", "core"),
            ("shredded cabbage", "supporting"),
            ("Japanese mayonnaise", "supporting"),
            ("lemon wedges", "optional"),
            ("pickled daikon", "optional"),
            ("nori strips", "optional"),
        ],
    },
    {
        "title": "Hiyashi Chuka (Cold Ramen Salad)",
        "ingredients": [
            ("ramen noodles", "core"),
            ("cucumber", "supporting"),
            ("tomato", "supporting"),
            ("ham or chicken strips", "supporting"),
            ("egg omelette strips", "supporting"),
            ("bean sprouts", "supporting"),
            ("hiyashi chuka dressing", "core"),
            ("nori strips", "optional"),
            ("sesame seeds", "optional"),
            ("karashi mustard", "optional"),
        ],
    },
    {
        "title": "Buta no Shogayaki (Ginger Pork)",
        "ingredients": [
            ("thinly sliced pork", "core"),
            ("grated ginger", "core"),
            ("soy sauce", "core"),
            ("mirin", "supporting"),
            ("sake", "supporting"),
            ("sugar", "supporting"),
            ("neutral frying oil", "supporting"),
            ("shredded cabbage", "optional"),
            ("steamed rice", "supporting"),
        ],
    },
]

\# 10 validation recipes
validation_recipes = [
    {
        "title": "Kinpira Gobo",
        "ingredients": [
            ("burdock root", "core"),
            ("carrot", "core"),
            ("sesame oil", "supporting"),
            ("soy sauce", "supporting"),
            ("sugar", "supporting"),
            ("mirin", "supporting"),
            ("sake", "supporting"),
            ("toasted sesame seeds", "optional"),
            ("shichimi togarashi", "optional"),
        ],
    },
    {
        "title": "Omurice (Japanese Omelette Rice)",
        "ingredients": [
            ("cooked rice", "core"),
            ("chicken thighs", "supporting"),
            ("onion", "supporting"),
            ("ketchup", "core"),
            ("eggs", "core"),
            ("butter", "supporting"),
            ("salt", "supporting"),
            ("black pepper", "supporting"),
            ("milk or cream", "supporting"),
            ("parsley", "optional"),
        ],
    },
    {
        "title": "Gyoza (Pan-Fried Dumplings)",
        "ingredients": [
            ("gyoza wrappers", "core"),
            ("ground pork", "core"),
            ("cabbage", "core"),
            ("garlic", "supporting"),
            ("ginger", "supporting"),
            ("scallions", "supporting"),
            ("soy sauce", "supporting"),
            ("sesame oil", "supporting"),
            ("salt", "supporting"),
            ("black pepper", "supporting"),
            ("vegetable oil", "supporting"),
            ("water", "supporting"),
            ("gyoza dipping sauce", "supporting"),
        ],
    },
    {
        "title": "Ochazuke (Salmon)",
        "ingredients": [
            ("cooked rice", "core"),
            ("hot green tea or dashi", "core"),
            ("grilled salted salmon", "core"),
            ("nori strips", "supporting"),
            ("sesame seeds", "optional"),
            ("wasabi", "optional"),
            ("pickled plum (umeboshi)", "optional"),
        ],
    },
    {
        "title": "Nabeyaki Udon",
        "ingredients": [
            ("udon noodles", "core"),
            ("dashi broth", "core"),
            ("soy sauce", "supporting"),
            ("mirin", "supporting"),
            ("chicken thighs", "supporting"),
            ("shiitake mushrooms", "supporting"),
            ("spinach", "supporting"),
            ("egg", "supporting"),
            ("tempura shrimp", "supporting"),
            ("kamaboko fish cake", "supporting"),
            ("scallions", "optional"),
        ],
    },
    {
        "title": "Yakitori (Negima Skewers)",
        "ingredients": [
            ("chicken thighs", "core"),
            ("negi (Japanese leek)", "core"),
            ("yakitori tare sauce", "core"),
            ("salt", "supporting"),
            ("shichimi togarashi", "optional"),
        ],
    },
    {
        "title": "Saba Shioyaki (Salt-Grilled Mackerel)",
        "ingredients": [
            ("mackerel fillets", "core"),
            ("salt", "core"),
            ("lemon wedges", "optional"),
            ("grated daikon radish", "optional"),
            ("soy sauce", "optional"),
            ("steamed rice", "supporting"),
        ],
    },
    {
        "title": "Oden (Simplified)",
        "ingredients": [
            ("daikon radish", "core"),
            ("boiled eggs", "supporting"),
            ("konnyaku", "supporting"),
            ("chikuwa fish cake", "supporting"),
            ("atsuage tofu", "supporting"),
            ("ganmodoki", "supporting"),
            ("dashi broth", "core"),
            ("soy sauce", "supporting"),
            ("mirin", "supporting"),
            ("karashi mustard", "optional"),
        ],
    },
    {
        "title": "Mentaiko Spaghetti",
        "ingredients": [
            ("spaghetti", "core"),
            ("mentaiko (spicy cod roe)", "core"),
            ("butter", "supporting"),
            ("soy sauce", "supporting"),
            ("heavy cream", "supporting"),
            ("nori strips", "optional"),
            ("shiso leaves", "optional"),
        ],
    },
    {
        "title": "Tonjiru (Pork and Vegetable Miso Soup)",
        "ingredients": [
            ("thinly sliced pork belly", "core"),
            ("daikon radish", "supporting"),
            ("carrot", "supporting"),
            ("konnyaku", "supporting"),
            ("burdock root", "supporting"),
            ("green onion", "optional"),
            ("dashi", "core"),
            ("miso paste", "core"),
            ("sesame oil", "supporting"),
        ],
    },
]